# Northwind Database — SQL + Pandas Business Analysis

This notebook connects to the Northwind SQLite database, runs the business queries defined in
`queries.sql`, loads the results into Pandas, explores them, and documents key business insights.

**Dataset:** [Northwind SQLite3](https://github.com/jpwhite3/northwind-SQLite3) — a SQLite port of
Microsoft's classic Northwind sample database (a small specialty food import/export company:
customers, orders, products, categories, employees, suppliers, shippers).

> ⚠️ **Setup required:** download `northwind.db` before running this notebook (see README.md for
> the link) and place it in the same folder as this notebook.


## 1. Connect to the database

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = "northwind.db"  # downloaded from the northwind-SQLite3 repo — see README.md

conn = sqlite3.connect(DB_PATH)
print("Connected to:", DB_PATH)

## 2. Load queries from `queries.sql`

Rather than duplicating SQL inside the notebook, we parse the queries straight out of
`queries.sql` using the `-- QUERY: <name>` markers, so the SQL lives in one place.

In [ ]:
import re

def load_queries(path="queries.sql"):
    with open(path, "r") as f:
        text = f.read()

    # Split on '-- QUERY: <name>' markers
    parts = re.split(r"--\s*QUERY:\s*(\w+)", text)
    # parts[0] is preamble before the first marker; then alternating name, sql, name, sql, ...
    queries = {}
    for i in range(1, len(parts), 2):
        name = parts[i].strip()
        sql = parts[i + 1].strip()
        queries[name] = sql
    return queries

queries = load_queries("queries.sql")
print("Loaded queries:", list(queries.keys()))

## 3. Quick schema / row-count overview

Before diving into the business questions, a quick look at table sizes gives a feel for the
dataset's scale.

In [ ]:
tables = ["Customers", "Orders", '"Order Details"', "Products", "Categories", "Employees", "Suppliers", "Shippers"]

for t in tables:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn).iloc[0, 0]
    print(f"{t:20s} {count:>6,} rows")

---
## 4. Business Question 1 — Top 10 Selling Products

In [ ]:
top_products = pd.read_sql_query(queries["top_10_products"], conn)
top_products

In [ ]:
print(top_products.dtypes)
print()
top_products.describe()

In [ ]:
ax = top_products.plot(x="ProductName", y="TotalRevenue", kind="barh", figsize=(8, 5), legend=False)
ax.invert_yaxis()
ax.set_xlabel("Total Revenue (£)")
ax.set_title("Top 10 Products by Revenue")
plt.tight_layout()
plt.show()

---
## 5. Business Question 2 — Top 10 Customers by Revenue

In [ ]:
top_customers = pd.read_sql_query(queries["top_10_customers"], conn)
top_customers

In [ ]:
ax = top_customers.plot(x="CompanyName", y="TotalRevenue", kind="barh", figsize=(8, 5), legend=False, color="darkorange")
ax.invert_yaxis()
ax.set_xlabel("Total Revenue (£)")
ax.set_title("Top 10 Customers by Revenue")
plt.tight_layout()
plt.show()

---
## 6. Business Question 3 — Monthly Sales Trend

In [ ]:
monthly_sales = pd.read_sql_query(queries["monthly_sales_trend"], conn)
monthly_sales.head(15)

In [ ]:
ax = monthly_sales.plot(x="OrderMonth", y="MonthlyRevenue", kind="line", marker="o", figsize=(11, 4), legend=False)
ax.set_ylabel("Revenue (£)")
ax.set_xlabel("Month")
ax.set_title("Monthly Revenue Trend")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

---
## 7. Business Question 4 — Best-Performing Product Categories

In [ ]:
category_performance = pd.read_sql_query(queries["category_performance"], conn)
category_performance

In [ ]:
ax = category_performance.plot(x="CategoryName", y="TotalRevenue", kind="bar", figsize=(8, 5), legend=False, color="seagreen")
ax.set_ylabel("Total Revenue (£)")
ax.set_title("Revenue by Product Category")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## 8. Business Question 5 — Customer Purchase Frequency

In [ ]:
purchase_frequency = pd.read_sql_query(queries["customer_purchase_frequency"], conn)
purchase_frequency.head(15)

In [ ]:
purchase_frequency["NumberOfOrders"].describe()

In [ ]:
ax = purchase_frequency["NumberOfOrders"].plot(kind="hist", bins=20, figsize=(8, 4), color="slateblue")
ax.set_xlabel("Number of Orders per Customer")
ax.set_title("Distribution of Customer Purchase Frequency")
plt.tight_layout()
plt.show()

---
## 9. Key Business Insights

The cell below **auto-generates** the five insights from whatever the live query results were —
so every time this notebook is re-run on updated data, the insight text updates with it.

In [ ]:
best_product = top_products.iloc[0]
best_customer = top_customers.iloc[0]
best_month = monthly_sales.loc[monthly_sales["MonthlyRevenue"].idxmax()]
worst_month = monthly_sales.loc[monthly_sales["MonthlyRevenue"].idxmin()]
best_category = category_performance.iloc[0]
avg_orders_per_customer = purchase_frequency["NumberOfOrders"].mean()
top_decile_cutoff = purchase_frequency["NumberOfOrders"].quantile(0.9)
frequent_customers = (purchase_frequency["NumberOfOrders"] >= top_decile_cutoff).sum()

insights = f"""
1. Best-selling product: "{best_product['ProductName']}" leads with £{best_product['TotalRevenue']:,.2f}
   in total revenue from {int(best_product['TotalUnitsSold']):,} units sold.

2. Top customer: "{best_customer['CompanyName']}" is the highest-revenue customer at
   £{best_customer['TotalRevenue']:,.2f} across {int(best_customer['TotalOrders'])} orders.

3. Seasonality: revenue peaked in {best_month['OrderMonth']} (£{best_month['MonthlyRevenue']:,.2f})
   and was lowest in {worst_month['OrderMonth']} (£{worst_month['MonthlyRevenue']:,.2f}),
   showing a {(best_month['MonthlyRevenue'] / worst_month['MonthlyRevenue'] - 1) * 100:.0f}%
   swing between the best and worst month.

4. Category concentration: "{best_category['CategoryName']}" is the top-performing category,
   generating £{best_category['TotalRevenue']:,.2f} in revenue from
   {int(best_category['ProductCount'])} products.

5. Purchase frequency: customers place {avg_orders_per_customer:.1f} orders on average, and the
   top 10% of customers (>= {top_decile_cutoff:.0f} orders) account for {frequent_customers} of the
   {len(purchase_frequency)} total customers — a useful segment for loyalty/retention campaigns.
"""
print(insights)

---
## 10. Close the connection

In [ ]:
conn.close()